In [44]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb

In [45]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train)

            id  Age  Sex  Chest pain type   BP  Cholesterol  FBS over 120  \
0            0   58    1                4  152          239             0   
1            1   52    1                1  125          325             0   
2            2   56    0                2  160          188             0   
3            3   44    0                3  134          229             0   
4            4   58    1                4  140          234             0   
...        ...  ...  ...              ...  ...          ...           ...   
629995  629995   56    0                1  110          226             0   
629996  629996   54    1                4  128          249             1   
629997  629997   67    1                4  130          275             0   
629998  629998   52    1                4  140          199             0   
629999  629999   51    0                2  130          199             0   

        EKG results  Max HR  Exercise angina  ST depression  Slope of ST  \

In [46]:
def feature_engineering(df):
  df_news = df.copy()

  df_news['HR_Ratio'] = df_news['Max HR'] / (220 - df_news['Age'] + 1e-5)
  df_news['RPP'] = df['BP'] * df_news['Max HR']
  df_news['BP_Age_Combination'] = df_news['Age'] * df_news['BP']
  df_news['Cholesterol_to_BP'] = df_news['Cholesterol'] / df_news['BP'] + 1e-5

  conditions_age = [
    (df['Age'] >= 20) & (df['Age'] <= 39),
    (df['Age'] >= 40) & (df['Age'] <= 59),
    (df['Age'] >= 60)
  ]

  conditions_bp = [
      (df['BP'] < 120),
      (df['BP'] >= 120) & (df['BP'] <= 129),
      (df['BP'] >= 130)
  ]

  choices = [0, 1, 2]

  df_news['Age_bins'] = np.select(conditions_age, choices)

  df_news['Is_ST_Depressed'] = (df_news['ST depression'] > 0).astype(int)
  df_news['Is_Thallium_wrong'] = (df_news['Thallium'] > 3).astype(int)

  df_news['BP_category'] = np.select(conditions_bp, choices)

  df_news['Thallium'] = df_news['Thallium'].map(df_news['Thallium'].value_counts(normalize=True))

  return df_news

In [47]:
train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

print(train_fe)

            id  Age  Sex  Chest pain type   BP  Cholesterol  FBS over 120  \
0            0   58    1                4  152          239             0   
1            1   52    1                1  125          325             0   
2            2   56    0                2  160          188             0   
3            3   44    0                3  134          229             0   
4            4   58    1                4  140          234             0   
...        ...  ...  ...              ...  ...          ...           ...   
629995  629995   56    0                1  110          226             0   
629996  629996   54    1                4  128          249             1   
629997  629997   67    1                4  130          275             0   
629998  629998   52    1                4  140          199             0   
629999  629999   51    0                2  130          199             0   

        EKG results  Max HR  Exercise angina  ...  Thallium  Heart Disease 

In [48]:
category_OneHot = ['Chest pain type', 'EKG results', 'Sex']
category_Scaler = ['RPP', 'BP_Age_Combination', 'Cholesterol_to_BP', 'Max HR', 'Age', 'BP']

OneHot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
scaler = StandardScaler()

data_encoder_train = OneHot.fit_transform(train_fe[category_OneHot])
data_encoder_test = OneHot.transform(test_fe[category_OneHot])

encoded_train = pd.DataFrame(data_encoder_train, columns=OneHot.get_feature_names_out(category_OneHot), index=train_fe.index)
encoded_test = pd.DataFrame(data_encoder_test, columns=OneHot.get_feature_names_out(category_OneHot), index=test_fe.index)

train_fe = train_fe.drop(columns=category_OneHot)
test_fe = test_fe.drop(columns=category_OneHot)

train_fe = pd.concat([train_fe, encoded_train], axis=1)
test_fe = pd.concat([test_fe, encoded_test], axis=1)

train_fe[category_Scaler] = scaler.fit_transform(train_fe[category_Scaler])
test_fe[category_Scaler] = scaler.transform(test_fe[category_Scaler])

In [49]:
X_train = train_fe.drop(columns='Heart Disease')
y_train = (train_fe['Heart Disease'] != 'Absence').astype(int)

model = xgb.XGBClassifier()

model.fit(X_train, y_train)

predict = model.predict(test_fe)

In [51]:
submission = pd.DataFrame({
    "id": test_fe['id'],
    'Heart Disease': predict
})

submission.to_csv("submission.csv", index=False)
print("\nSUBMISSION SAVED: 'submission.csv'")


SUBMISSION SAVED: 'submission.csv'
